# Verification of the marginal time-cost transform

**Supplementary notebook to** *Marginal time-cost analysis of drug release*

Jehad Nasereddin — Department of Pharmaceutical Technology and Cosmetics, Faculty of Pharmacy,
Middle East University, Amman 11831, Jordan — `j.nasereddin@meu.edu.jo`

---

This notebook reproduces every quantitative claim in the article and in Supplementary Material S1.
The seven checks below are numbered to match S1 §S3.

Each check is one of two kinds, and the distinction is the point of the notebook:

| Kind | What "verified" means | Expected residual |
|---|---|---|
| **Exact identity** | Established symbolically; the residual is zero by construction | `0` |
| **Numerical approximation** | Finite-precision evaluation; a residual is expected | Bounded by a stated error budget |

Reporting an exact identity as though it were an approximation understates it; reporting an
approximation without its error budget overstates it. Both failure modes are avoided here by
labelling every check.

**Requirements:** `sympy`, `numpy`. Symbolic work is exact; numerical work is double precision
(machine epsilon $\varepsilon \approx 2.2 \times 10^{-16}$).

**Notation.** $Q$ is cumulative amount released, $t(Q)$ the time to reach it, and
$c(Q) = \mathrm{d}t/\mathrm{d}Q$ the marginal time-cost — a time per unit released.

In [ ]:
import numpy as np
import sympy as sp

sp.init_printing(use_unicode=True)
print("sympy", sp.__version__, "| numpy", np.__version__)
print("machine epsilon:", np.finfo(float).eps)

# Shared symbols
u, Q, t = sp.symbols('u Q t', positive=True)
kH, k0, k = sp.symbols('k_H k_0 k', positive=True)
n, a, b = sp.symbols('n a b', positive=True)
Qinf = sp.symbols('Q_inf', positive=True)

PASS, FAIL = "PASS", "FAIL"

def report(name, residual, kind, budget=None):
    # Print a uniform verdict line for a check.
    if kind == "exact":
        ok = (sp.simplify(residual) == 0)
        print(f"[{PASS if ok else FAIL}] {name}\n       exact identity; residual = {sp.simplify(residual)}")
    else:
        ok = abs(residual) <= budget
        print(f"[{PASS if ok else FAIL}] {name}\n       numerical; residual = {residual:.3e}  (budget {budget:.1e})")
    return ok

results = {}

---
## Check 1 — Recovery of the release curve — *exact*

**Claim (article §2.2).** The transform loses nothing: $t(Q) = \int_0^Q c(u)\,\mathrm{d}u$ returns the
release curve exactly.

For Higuchi, $Q = k_H\sqrt{t}$, the cost is $c(u) = 2u/k_H^2$. Integrating should return $t$ *identically*
— for every $k_H$ and every $t$, not approximately.

In [ ]:
c_hig = 2*u / kH**2
antider = sp.integrate(c_hig, (u, 0, Q))
recovered = sp.simplify(antider.subs(Q, kH*sp.sqrt(t)))

print("c(u)                =", c_hig)
print("∫₀^Q c(u) du        =", sp.simplify(antider))
print("with Q = k_H·√t     =", recovered)

results['check1_symbolic'] = report("Check 1 — recovery", recovered - t, "exact")

**Numerical confirmation.** Note $c(Q) = 2Q/k_H^2$ is *linear* in $Q$, and the trapezoidal rule
integrates linear functions exactly. So the only possible error is floating-point round-off — the
residual should be at machine level, not at discretisation level.

In [ ]:
kHv = 12.0
print(f"{'t':>6} {'Q = k_H√t':>14} {'trapz ∫':>18} {'residual |t-∫|':>16}")
worst = 0.0
for tv in [1.0, 4.0, 9.0]:
    Qv = kHv*np.sqrt(tv)
    grid = np.linspace(0.0, Qv, 200_001)
    integ = np.trapezoid(2*grid/kHv**2, grid)
    res = abs(tv - integ)
    worst = max(worst, res)
    print(f"{tv:6.1f} {Qv:14.6f} {integ:18.12f} {res:16.3e}")

results['check1_numeric'] = report("Check 1 — numerical recovery", worst, "numeric", budget=1e-9)

---
## Check 2 — Higuchi is the $n=\tfrac12$ case of Korsmeyer–Peppas — *exact*

**Claim (article §4.5).** Higuchi is not an independent model but a special case of the power law.
This matters to the argument: two "rival" models that are algebraically nested cannot be
discriminated by comparing their fits.

General Peppas cost: $c(Q) = \dfrac{Q^{(1-n)/n}}{n\,k^{1/n}}$. Setting $n = \tfrac12$, $k = k_H$ should
return $2Q/k_H^2$ exactly.

In [ ]:
c_peppas = Q**((1-n)/n) / (n * k**(1/n))
sub = sp.powsimp(sp.simplify(c_peppas.subs({n: sp.Rational(1,2), k: kH})), force=True)
c_hig_direct = 2*Q/kH**2

print("general Peppas cost   c(Q) =", c_peppas)
print("  exponent (1-n)/n at n=½  =", sp.simplify(((1-n)/n).subs(n, sp.Rational(1,2))))
print("  prefactor n·k^(1/n)      =", sp.simplify((n*k**(1/n)).subs({n: sp.Rational(1,2), k: kH})))
print("  => c(Q) =", sub)
print("Higuchi derived directly   =", c_hig_direct)

results['check2'] = report("Check 2 — Higuchi ⊂ Peppas", sub - c_hig_direct, "exact")

---
## Check 3 — Hixson–Crowell derivative — *numerical, with error budget*

**Claim (article §4.5).** For the cube-root law, $\mathrm{d}Q/\mathrm{d}t = 3\kappa(Q_\infty - Q)^{2/3}$.

This one involves a fractional power, so it is checked numerically by central difference. **A residual
is expected here** and its size is predicted in advance:

- truncation error $O(h^2) = 10^{-12}$
- round-off error $O(\varepsilon/h) = 10^{-16}/10^{-6} = 10^{-10}$

so the floor is around $10^{-9}$. A residual at that scale confirms the derivative; a residual much
*below* it would suggest the test is not actually exercising the arithmetic.

In [ ]:
Qinf_v, kap_v, h = 100.0, 0.15, 1e-6
tmax = Qinf_v**(1/3) / kap_v

def Q_of_t(tv):
    return Qinf_v - (Qinf_v**(1/3) - kap_v*tv)**3

print(f"Q_inf = {Qinf_v}, κ = {kap_v}, release completes at t = {tmax:.4f} min")
print(f"central difference [Q(t+h) − Q(t−h)] / 2h,  h = {h:g}\n")
print(f"{'t':>6} {'Q(t)':>10} {'numeric dQ/dt':>16} {'analytic dQ/dt':>16} {'abs diff':>11} {'rel diff':>10}")

worst = 0.0
for tv in [0.5, 3.0, 10.0, 25.0, 30.0]:
    num = (Q_of_t(tv+h) - Q_of_t(tv-h)) / (2*h)
    ana = 3*kap_v*(Qinf_v - Q_of_t(tv))**(2/3)
    worst = max(worst, abs(num-ana))
    print(f"{tv:6.1f} {Q_of_t(tv):10.5f} {num:16.9f} {ana:16.9f} {abs(num-ana):11.2e} {abs(num-ana)/ana:10.2e}")

results['check3'] = report("Check 3 — Hixson–Crowell derivative", worst, "numeric", budget=1e-7)

---
## Check 4 — Weibull reduces to first-order at $\beta = 1$ — *exact*

**Claim (article §4.5).** Weibull and first-order are not independent either. At $\beta = 1$,
$\tau = 1/k$, the Weibull cost collapses onto the first-order cost — which is why both land in Type III.

In [ ]:
tau, beta, f = sp.symbols('tau beta f', positive=True)

dtdf = (tau/beta) * (-sp.log(1-f))**(1/beta - 1) / (1-f)
print("Weibull  dt/df =", dtdf)

step1 = sp.simplify(dtdf.subs(beta, 1))
print("at β = 1: (−ln(1−f))^0 = 1, so dt/df =", step1)

c_weib = sp.simplify((step1/Qinf).subs({tau: 1/k, f: Q/Qinf}))
c_first = 1/(k*(Qinf - Q))

print("c(Q) = (1/Q_inf)·dt/df, with τ=1/k, f=Q/Q_inf  =>", sp.simplify(c_weib))
print("first-order derived directly                    =", c_first)

results['check4'] = report("Check 4 — Weibull(β=1) = first-order", c_weib - c_first, "exact")

---
## Check 5 — Marginal-to-average ratio — *exact*

**Claim (article §2.7).** For a power law $t = aQ^b$, the ratio of marginal cost to average cost is
exactly $b$, independent of $Q$.

This is what makes $b$ measurable without fitting: it is a ratio of two quantities a dissolution
experiment already yields.

In [ ]:
t_pow = a*Q**b
marg = sp.diff(t_pow, Q)
avg = t_pow/Q
ratio = sp.simplify(marg/avg)

print("t(Q)              =", t_pow)
print("marginal  dt/dQ   =", marg)
print("average   t/Q     =", sp.simplify(avg))
print("ratio             =", ratio, "  <- exactly b, independent of Q")

results['check5_symbolic'] = report("Check 5 — marginal/average = b", ratio - b, "exact")

print("\nNumerical spot-check, a = 0.35, b = 1.4:")
av, bv = 0.35, 1.4
print(f"{'Q':>5} {'marginal':>18} {'average':>18} {'ratio':>12} {'|ratio−b|':>11}")
worst = 0.0
for Qv in [10, 40, 80]:
    m = av*bv*Qv**(bv-1)
    g = av*Qv**bv/Qv
    worst = max(worst, abs(m/g - bv))
    print(f"{Qv:5d} {m:18.9f} {g:18.9f} {m/g:12.9f} {abs(m/g-bv):11.2e}")

results['check5_numeric'] = report("Check 5 — numerical ratio", worst, "numeric", budget=1e-12)

---
## Check 6 — Mean dissolution time is a functional of the cost curve — *exact, and general*

**Claim (article §4.7).** MDT is not an independent construct; it is a weighted integral of $c$:

$$\mathrm{MDT} = \frac{1}{Q_\infty}\int_0^{Q_\infty} (Q_\infty - u)\,c(u)\,\mathrm{d}u$$

The general result for any power law is derived first, then two special cases are worked explicitly.
The ratio $\mathrm{MDT}/T_\infty = 1/(b+1)$ is what makes §4.8's "MDT is blind to the tail" argument
work: it depends only on $b$, and so cannot see where in the curve the time was spent.

In [ ]:
c_gen = a*b*u**(b-1)
mdt_gen = sp.simplify(sp.integrate((Qinf - u)*c_gen, (u, 0, Qinf)) / Qinf)
Tinf_gen = a*Qinf**b
ratio_gen = sp.simplify(mdt_gen/Tinf_gen)

print("GENERAL: for any power law t = a·Q^b, c = a·b·Q^(b−1)")
print("   MDT        =", mdt_gen)
print("   T_inf      =", Tinf_gen)
print("   MDT/T_inf  =", ratio_gen, "   <- 1/(b+1), exactly")
print("   zero-order (b=1):", sp.simplify(ratio_gen.subs(b, 1)))
print("   Higuchi    (b=2):", sp.simplify(ratio_gen.subs(b, 2)), "  <- the 1/3 quoted in article §4.7")

results['check6_general'] = report("Check 6 — MDT/T_inf = 1/(b+1)", ratio_gen - 1/(b+1), "exact")

In [ ]:
print("--- Zero-order worked explicitly: k0 = 2, Q_inf = 100, c = 1/k0 = 0.5 ---")
w = sp.integrate((100-u)*sp.Rational(1,2), (u, 0, 100))/100
d = sp.integrate(u/2, (u, 0, 100))/100
print("   weighted form (1/100)∫(100−u)·½ du =", w, "=", float(w))
print("   direct form   (1/100)∫t(Q) dQ      =", d, "=", float(d))
print("   theory        T_inf/2               =", (100.0/2.0)/2)
r1 = report("Check 6 — zero-order MDT agreement", w - d, "exact")

print("\n--- Higuchi worked explicitly: k_H = 12, Q_inf = 100, c = 2Q/144 ---")
w2 = sp.integrate((100-u)*(2*u/144), (u, 0, 100))/100
d2 = sp.integrate(u**2/144, (u, 0, 100))/100
Tinf2 = sp.Rational(100**2, 144)
print("   weighted form =", w2, "=", float(w2))
print("   direct form   =", d2, "=", float(d2))
print("   T_inf         =", Tinf2, "=", float(Tinf2))
print("   MDT/T_inf     =", sp.simplify(w2/Tinf2), "=", float(w2/Tinf2), " <- exactly 1/3")
r2 = report("Check 6 — Higuchi MDT agreement", w2 - d2, "exact")
r3 = report("Check 6 — Higuchi MDT/T_inf = 1/3", sp.simplify(w2/Tinf2) - sp.Rational(1,3), "exact")

results['check6_worked'] = r1 and r2 and r3

---
## Check 7 — Tail share $1 - T_{90}/T_{100}$ — *exact*

**Claim (article §4.8).** For a power law, $T_X = a(X\%\cdot Q_\infty)^b$, so $T_{90}/T_{100} = 0.9^b$ —
the fraction of total time spent on the last 10% of the dose depends only on $b$.

This is the tail-expense measure that requires no fitting.

In [ ]:
print(f"{'model':16s} {'b':>8} {'0.9^b':>10} {'1 − T90/T100':>14}")
for name, bval in [("zero-order", 1.0), ("Peppas n=0.7", 1/0.7), ("Higuchi", 2.0)]:
    print(f"{name:16s} {bval:8.4f} {0.9**bval:10.6f} {1-0.9**bval:14.6f}")

# symbolic confirmation that the ratio is 0.9^b and carries no other dependence
X = sp.Rational(9, 10)
TX = a*(X*Qinf)**b
T100 = a*Qinf**b
share = sp.simplify(TX/T100)
print("\nsymbolic: T90/T100 =", share, "  <- depends on b alone; a and Q_inf cancel")

results['check7'] = report("Check 7 — tail share = 0.9^b", share - X**b, "exact")

---
## Summary

In [ ]:
print("=" * 62)
for name, ok in results.items():
    print(f"  {'PASS' if ok else 'FAIL'}   {name}")
print("=" * 62)
n_pass = sum(1 for v in results.values() if v)
print(f"{n_pass} / {len(results)} checks passed")
if all(results.values()):
    print("\nAll verification claims in the article and S1 reproduce.")
else:
    print("\nAT LEAST ONE CHECK FAILED — do not submit.")